# Argmax for LLM Token Selection with Custom Reduction

This notebook picks tokens from `[B, V]` logits with `cudnn.reduction_modifiers.argmax`, a ready-made custom reduction:

1. **Greedy decoding** — the next token is the argmax of the BF16 logits.
2. **NaN rules** — `TORCH` (default, as `torch.argmax`), `XLA` and `IGNORE_NAN`, for argmax and argmin.
3. **Speculative-decoding verify** — the target argmax of each draft position.
4. **Masked argmax and Gumbel-max sampling** — pointwise operations compute the argmax input in the same graph.

Each result is checked against PyTorch.

## Prerequisites and Setup

This notebook requires cuDNN 9.29 or later and an NVIDIA **data center Blackwell** GPU: the custom reduction runs on the TensorIR mem-bound engine. On other devices the cuDNN cells are skipped.

**Environment setup:**

- **Option A — pip install:**
  ```bash
  pip install nvidia-cudnn-frontend
  ```
- **Option B — set paths manually:**
  ```bash
  export LD_LIBRARY_PATH=/path/to/cudnn/lib:${LD_LIBRARY_PATH}
  export PYTHONPATH=/path/to/cudnn_frontend/build_python:${PYTHONPATH}
  ```

In [ ]:
# get_ipython().system('pip install nvidia-cudnn-cu13')
# get_ipython().system('pip install nvidia-cudnn-frontend')
# get_ipython().system('pip3 install --pre torch --index-url https://download.pytorch.org/whl/nightly/cu130')

In [ ]:
import warnings

warnings.filterwarnings("ignore", message="Failed to initialize NumPy")

import torch
import cudnn

print("cuDNN backend version:", cudnn.backend_version())

has_cuda = torch.cuda.is_available()
_major, _minor = torch.cuda.get_device_capability() if has_cuda else (0, 0)
_sm = _major * 10 + _minor
compiled_cudnn_version = getattr(cudnn._pybind_module, "_compiled_cudnn_version", 0)
is_supported = has_cuda and 100 <= _sm < 110 and compiled_cudnn_version >= 92900 and cudnn.backend_version() >= 92900
if not is_supported:
    print("Skipping the cuDNN cells: they need frontend headers and runtime cuDNN 9.29 or later, and data center Blackwell.")

## Greedy Decoding

`argmax` takes two streams: the values and the index of each element (`0 ... V-1` along the vocabulary). It returns the
`reducer`, `reducer_semantic_key` and `identity_values` arguments of `custom_reduction`. Only the index output is read.

In [ ]:
B, V = 8, 129280


def argmax_graph(handle, logits_like, kind="argmax", rule="torch", producer=None, extra_inputs=()):
    """A graph that reduces logits (or producer(graph, logits, *extra)) along the vocabulary; returns its tensors."""
    graph = cudnn.pygraph(
        handle=handle,
        io_data_type=cudnn.data_type.FLOAT,
        intermediate_data_type=cudnn.data_type.FLOAT,
        compute_data_type=cudnn.data_type.FLOAT,
    )
    logits = graph.tensor_like(logits_like)
    extra = [graph.tensor_like(t) for t in extra_inputs]
    for tensor, like in zip(extra, extra_inputs):
        if like.dtype == torch.bool:
            tensor.set_data_type(cudnn.data_type.BYTE_BOOLEAN)  # torch.bool holds one byte per element
    index = graph.tensor(dim=list(logits_like.shape), stride=list(logits_like.stride()), data_type=cudnn.data_type.INT32)
    if producer is None:
        values, value_type = logits, logits.get_data_type()
    else:
        values, value_type = producer(graph, logits, *extra), cudnn.data_type.FLOAT
    helper = getattr(cudnn.reduction_modifiers, kind)(value_type, rule=rule)
    _, token = graph.custom_reduction([values, index], reduction_dims=[1], **helper)
    token.set_output(True)
    graph.build([cudnn.heur_mode.A])
    return graph, logits, extra, index, token


def run(handle, built, logits_gpu, extra_gpu=()):
    graph, logits, extra, index, token = built
    rows, vocabulary = logits_gpu.shape
    feeds = {logits: logits_gpu, index: torch.arange(vocabulary, dtype=torch.int32, device="cuda").expand(rows, vocabulary).contiguous()}
    feeds.update(zip(extra, extra_gpu))
    feeds[token] = torch.full((rows, 1), -1, dtype=torch.int32, device="cuda")
    workspace = torch.empty(graph.get_workspace_size(), device="cuda", dtype=torch.uint8)
    torch.cuda.synchronize()
    graph.execute(feeds, workspace, handle=handle)
    torch.cuda.synchronize()
    return feeds[token].view(-1).long()


if is_supported:
    handle = cudnn.create_handle()
    torch.manual_seed(0)
    logits_gpu = (torch.randn(B, V, device="cuda") * 3).to(torch.bfloat16)
    next_token = run(handle, argmax_graph(handle, logits_gpu), logits_gpu)
    assert torch.equal(next_token, torch.argmax(logits_gpu, dim=1))
    print("next tokens:", next_token.tolist())
    print("PASSED: greedy argmax matches torch.argmax.")

## NaN Rules

Every rule gives the lowest index among equal values, and `-0` equals `+0`:

| Rule | Row with NaN | Matches |
|---|---|---|
| `"torch"` (default) | the first NaN | `torch.argmax`, `torch.argmin` |
| `"xla"` | one of the NaN positions (with two or more NaN, which one is not specified) | the XLA comparator |
| `"ignore_nan"` | the best number; a row of NaN only gives its first position | a NaN-ignoring argmax |

The rows below hold NaN, ties, signed zeros and infinities.

In [ ]:
def reference(x, kind, rule, got):
    """Rows whose index breaks the rule."""
    is_max = kind == "argmax"
    in_range = (got >= 0) & (got < x.shape[1])  # an unwritten index is out of range
    if not bool(in_range.all()):
        return int((~in_range).sum())
    torch_index = torch.argmax(x, dim=1) if is_max else torch.argmin(x, dim=1)
    if rule == "torch":
        return int((got != torch_index).sum())
    nan = torch.isnan(x)
    rows = torch.arange(x.shape[0], device=x.device)
    if rule == "xla":
        return int((~torch.where(nan.any(dim=1), nan[rows, got], got == torch_index)).sum())
    numbers = x.float().masked_fill(nan, float("-inf") if is_max else float("inf"))
    best = numbers.amax(dim=1, keepdim=True) if is_max else numbers.amin(dim=1, keepdim=True)
    first_best = torch.argmax(((numbers == best) & ~nan).to(torch.uint8), dim=1)
    return int((got != torch.where((~nan).any(dim=1), first_best, torch.zeros_like(first_best))).sum())


if is_supported:
    nan, inf = float("nan"), float("inf")
    special = torch.randn(8, V, device="cuda") * 3
    special[0, 100] = nan  # one NaN
    special[1, [5, 6]] = nan  # two NaN
    special[2, :] = nan  # only NaN
    special[3, [7, 9]] = 50.0  # a tie
    special[4, :] = -1.0
    special[4, [11, 3]] = torch.tensor([0.0, -0.0], device="cuda")  # signed zeros
    special[5, :] = -inf
    special[6, [2, 4]] = inf
    special[7, 0], special[7, 1:] = nan, -inf
    special = special.to(torch.bfloat16)
    for kind in ("argmax", "argmin"):
        for rule in ("torch", "xla", "ignore_nan"):
            got = run(handle, argmax_graph(handle, special, kind, rule), special)
            mismatches = reference(special, kind, rule, got)
            print(f"{kind:7s} {rule:11s} indices={got.tolist()} mismatches={mismatches}")
            assert mismatches == 0
    print("PASSED: each rule matches its reference.")

## Speculative-Decoding Verify

The target model scores `B` sequences with `k` draft tokens each; one argmax over the `[B * (k + 1), V]` target logits
gives the target token of each position. A sequence accepts its draft tokens up to the first one that differs.

In [ ]:
if is_supported:
    k = 3
    target_logits = (torch.randn(B * (k + 1), V, device="cuda") * 3).to(torch.bfloat16)
    target = run(handle, argmax_graph(handle, target_logits), target_logits).view(B, k + 1)
    assert torch.equal(target.view(-1), torch.argmax(target_logits, dim=1))
    draft = target[:, :k].clone()
    draft[torch.arange(B), torch.arange(B) % k] += 1  # sequence b differs at position b % k
    accepted = (draft == target[:, :k]).int().cumprod(dim=1).sum(dim=1)
    print("accepted draft tokens per sequence:", accepted.tolist())
    assert accepted.tolist() == [b % k for b in range(B)]
    print("PASSED: verify matches torch.argmax.")

## Masked Argmax and Gumbel-Max Sampling

Pointwise operations of the same graph produce the argmax input:

- **masked argmax:** `select(logits, -inf, allowed)` with a boolean `allowed` tensor;
- **Gumbel-max:** `logits / T + g`, with one temperature per row and the noise `g = -log(-log(u))` as an input tensor
  (a graph has no random number generator).

In [ ]:
def masked(graph, logits, allowed):
    minus_infinity = graph.tensor_scalar(float("-inf"), cudnn.scalar_type.COMPILE_TIME_CONST)
    return graph.binary_select(logits, minus_infinity, allowed)


def gumbel(graph, logits, temperature, noise):
    return graph.add(graph.div(logits, temperature), noise)


if is_supported:
    allowed = torch.rand(B, V, device="cuda") < 0.3
    got = run(handle, argmax_graph(handle, logits_gpu, producer=masked, extra_inputs=[allowed]), logits_gpu, [allowed])
    want = torch.argmax(torch.where(allowed, logits_gpu.float(), torch.tensor(float("-inf"), device="cuda")), dim=1)
    assert torch.equal(got, want)
    print("masked argmax:", got.tolist())

    temperature = torch.rand(B, 1, device="cuda") + 0.5
    noise = -torch.log(-torch.log(torch.rand(B, V, device="cuda").clamp(1e-20, 1 - 1e-7)))
    built = argmax_graph(handle, logits_gpu, producer=gumbel, extra_inputs=[temperature, noise])
    got = run(handle, built, logits_gpu, [temperature, noise])
    assert torch.equal(got, torch.argmax(logits_gpu.float() / temperature + noise, dim=1))
    print("Gumbel-max samples:", got.tolist())
    print("PASSED: masked argmax and Gumbel-max match torch.")

In [ ]:
if is_supported:
    cudnn.destroy_handle(handle)